# EM-DAT Exploratory Data Analysis (EDA) Notebook
### Reusable analysis template for visualizing missing values, distributions, and correlations in EM-DAT datasets.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
matplotlib_inline_format = 'retina'
%matplotlib inline

ModuleNotFoundError: No module named 'pandas'

## 1. Load the Dataset
Point to the EM-DAT CSV file stored in the raw data directory.

In [ ]:
CSV_PATH = os.path.join("..", "data", "raw", "public_emdat_custom_request_2026-06-16_b4cec7bb-ec36-4c87-9762-f7cc13e97076.csv")

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Target EMDAT CSV not found at: {CSV_PATH}. Check paths.")

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig')
print(f"Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns.")

## 2. Identify Numerical Columns
Extract columns containing numerical data scales for targeted statistical inspection.

In [ ]:
numerical_cols = [
    'Magnitude',
    'Total Deaths',
    'No. Injured',
    'No. Affected',
    'No. Homeless',
    'Total Affected',
    'Reconstruction Costs (\'000 US$)',
    'Reconstruction Costs, Adjusted (\'000 US$)',
    'Insured Damage (\'000 US$)',
    'Insured Damage, Adjusted (\'000 US$)',
    'Total Damage (\'000 US$)',
    'Total Damage, Adjusted (\'000 US$)',
    'CPI'
]

# Filter down to columns actually present in the dataframe
numerical_cols = [col for col in numerical_cols if col in df.columns]
print("Numerical columns identified for inspection:", numerical_cols)

## 3. Visualize Missing Values
Determine the density of missing values per numerical column to inform imputation choices.

In [ ]:
missing_counts = df[numerical_cols].isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing_counts,
    'Percentage (%)': missing_percentages
}).sort_values(by='Percentage (%)', ascending=False)

print(missing_df)

# Plot missing percentages
plt.figure(figsize=(10, 6))
sns.barplot(x=missing_df['Percentage (%)'], y=missing_df.index, palette='crest')
plt.title('Percentage of Missing Values in Numerical EMDAT Columns', fontsize=14)
plt.xlabel('Null Percentage (%)', fontsize=12)
plt.ylabel('Column Name', fontsize=12)
plt.xlim(0, 100)
plt.tight_layout()
plt.show()

## 4. Visualize Numerical Distributions
Use logarithmic scaling on highly skewed metrics (like casualties and damages) to review variance.

In [ ]:
for col in numerical_cols:
    # Skip columns that are completely empty
    if df[col].isnull().all():
        print(f"Skipping {col}: all values are null.")
        continue
        
    # Setup subplots (Histogram & Boxplot side-by-side)
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Drop nulls for plotting
    data = df[col].dropna()
    
    # In emergency data, 0 vs millions distribution values require optional log scale check
    skew = data.skew()
    use_log = skew > 3.0 and (data > 0).any()
    
    if use_log:
        # Plot log-scale data
        log_data = np.log10(data[data > 0])
        sns.histplot(log_data, kde=True, ax=axes[0], color='teal')
        axes[0].set_title(f'{col} Distribution (Log10 Transformed)', fontsize=12)
        axes[0].set_xlabel(f'Log10({col})')
        
        sns.boxplot(x=log_data, ax=axes[1], color='teal')
        axes[1].set_title(f'{col} Outlier Spread (Log10)', fontsize=12)
        axes[1].set_xlabel(f'Log10({col})')
    else:
        # Standard Scale
        sns.histplot(data, kde=True, ax=axes[0], color='indigo')
        axes[0].set_title(f'{col} Standard Distribution', fontsize=12)
        axes[0].set_xlabel(col)
        
        sns.boxplot(x=data, ax=axes[1], color='indigo')
        axes[1].set_title(f'{col} Standard Outlier Spread', fontsize=12)
        axes[1].set_xlabel(col)
        
    plt.suptitle(f"Feature Profiling: {col} (Skewness: {skew:.2f})", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## 5. Correlation Matrix & Heatmap
Compute correlation matrices. Since the data is non-normally distributed and ordinal severity applies, we evaluate both **Pearson** (linear) and **Spearman** (rank-based) coefficients.

In [ ]:
# Filter down numerical columns with sufficient observations (>10% valid)
valid_numerical_cols = [col for col in numerical_cols if missing_percentages[col] < 90.0]

pearson_corr = df[valid_numerical_cols].corr(method='pearson')
spearman_corr = df[valid_numerical_cols].corr(method='spearman')

# Plot Pearson linear correlation heatmap
plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))
sns.heatmap(pearson_corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Pearson Linear Correlation Heatmap (EM-DAT)', fontsize=14)
plt.tight_layout()
plt.show()

# Plot Spearman rank correlation heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(spearman_corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Spearman Rank Correlation Heatmap (EM-DAT)', fontsize=14)
plt.tight_layout()
plt.show()